# B-cell epitope vs InterPLM SAE features on ESM2-8M

This notebook implements the workflow:

1. Load the curated linear B-cell epitope table.
2. Fetch full protein sequences with `requests`.
3. Verify epitope sequence and coordinates against the full sequence.
4. Build residue-level epitope labels per protein.
5. Run full protein sequences through `facebook/esm2_t6_8M_UR50D`.
6. Encode per-residue hidden states with a pretrained InterPLM SAE.
7. Compare binary feature-on residues to binary epitope residues across thresholds `[0.15, 0.5, 0.6, 0.8]`.

InterPLM README notes used here:

- Pretrained SAEs are available for ESM2-8M layers `1, 2, 3, 4, 5, 6`.
- `load_sae_from_hf(plm_model="esm2-8m", plm_layer=4)` is the reference loading path.
- The README walkthrough uses layer `4` as the middle layer for ESM2-8M, so this notebook defaults to `SAE_LAYER = 4`.

Important dataset note:

The CSV column named `uniprot` is not purely UniProt. It contains a mix of UniProt-like accessions and RefSeq/GenBank-style protein accessions. This notebook therefore uses a UniProt-first fetch strategy with an NCBI fallback.

In [17]:
from __future__ import annotations

import json
import math
import os
import re
import sys
import time
from collections import defaultdict
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Tuple

import numpy as np
import pandas as pd
import requests
import torch
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from tqdm.auto import tqdm

PROJECT_ROOT = Path.cwd()
INTERPLM_ROOT = PROJECT_ROOT / "InterPLM"
if str(INTERPLM_ROOT) not in sys.path:
    sys.path.insert(0, str(INTERPLM_ROOT))

from interplm.embedders.esm import ESM
from interplm.sae.inference import load_sae_from_hf
from interplm.utils import get_device

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)
torch.set_grad_enabled(False)

DATA_CSV = PROJECT_ROOT / "curated_b_cell_epitope_table_allergic_only_all_host.csv"
CACHE_DIR = PROJECT_ROOT / "analysis_cache"
OUTPUT_DIR = PROJECT_ROOT / "analysis_outputs"
CACHE_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

SAE_LAYER = 4
MODEL_NAME = "facebook/esm2_t6_8M_UR50D"
FEATURE_THRESHOLDS = [0.15, 0.3, 0.5, 0.6, 0.8]
MAX_SEQUENCE_LENGTH = 1022
BATCH_SIZE = 8
RESCUE_UNIQUE_MISMATCHES = True
SEQUENCE_CACHE_PATH = CACHE_DIR / "protein_sequence_cache.json"
VALIDATION_PATH = OUTPUT_DIR / "b_cell_allergic_epitope_sequence_validation.csv"
METRICS_PATH = OUTPUT_DIR / f"b_cell_allergic_epitope_feature_metrics_layer_{SAE_LAYER}.csv"
THRESHOLD_SUMMARY_PATH = OUTPUT_DIR / f"b_cell_allergic_epitope_threshold_summary_layer_{SAE_LAYER}.csv"
PROTEIN_SUMMARY_PATH = OUTPUT_DIR / f"b_cell_allergic_epitope_protein_summary_layer_{SAE_LAYER}.csv"
MERGED_EPITOPE_SEQUENCE_PATH = OUTPUT_DIR / "curated_b_cell_epitope_table_allergic_only_all_host_with_sequence.csv"

device = get_device()
device


'cpu'

In [18]:
UNIPROT_REST_FASTA = "https://rest.uniprot.org/uniprotkb/{accession}.fasta"
NCBI_EFETCH = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi"
UNIPROT_ACCESSION_RE = re.compile(
    r"^(?:[OPQ][0-9][A-Z0-9]{3}[0-9]|[A-NR-Z][0-9](?:[A-Z][A-Z0-9]{2}[0-9]){1,2})(?:-[0-9]+)?$"
)
AA_ONLY_RE = re.compile(r"[^A-Z]")


def build_session() -> requests.Session:
    session = requests.Session()
    retry = Retry(
        total=5,
        backoff_factor=1.0,
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=["GET"],
    )
    adapter = HTTPAdapter(max_retries=retry)
    session.mount("https://", adapter)
    session.mount("http://", adapter)
    session.headers.update({"User-Agent": "XAllergen-epitope-analysis/1.0"})
    return session


def parse_fasta(text: str) -> str:
    lines = [line.strip() for line in text.splitlines() if line.strip()]
    if not lines:
        return ""
    if lines[0].startswith(">"):
        lines = lines[1:]
    return "".join(lines).upper()


def strip_version(accession: str) -> str:
    accession = str(accession).strip()
    return re.sub(r"\.\d+$", "", accession)


def classify_accession(accession: str) -> str:
    cleaned = strip_version(accession)
    if UNIPROT_ACCESSION_RE.fullmatch(cleaned):
        return "uniprot"
    return "ncbi"


def fetch_uniprot_sequence(session: requests.Session, accession: str, timeout: int = 30) -> Optional[str]:
    cleaned = strip_version(accession)
    response = session.get(UNIPROT_REST_FASTA.format(accession=cleaned), timeout=timeout)
    if response.status_code == 404:
        return None
    response.raise_for_status()
    sequence = parse_fasta(response.text)
    return sequence or None


def fetch_ncbi_sequence(session: requests.Session, accession: str, timeout: int = 30) -> Optional[str]:
    params = {
        "db": "protein",
        "id": accession,
        "rettype": "fasta",
        "retmode": "text",
    }
    response = session.get(NCBI_EFETCH, params=params, timeout=timeout)
    if response.status_code == 404:
        return None
    response.raise_for_status()
    text = response.text.strip()
    if text.startswith("Error") or not text:
        return None
    sequence = parse_fasta(text)
    return sequence or None


def fetch_sequence(session: requests.Session, accession: str) -> Tuple[Optional[str], Optional[str]]:
    source = classify_accession(accession)
    if source == "uniprot":
        sequence = fetch_uniprot_sequence(session, accession)
        if sequence is not None:
            return sequence, "uniprot"
        sequence = fetch_ncbi_sequence(session, accession)
        if sequence is not None:
            return sequence, "ncbi_fallback"
    else:
        sequence = fetch_ncbi_sequence(session, accession)
        if sequence is not None:
            return sequence, "ncbi"
        cleaned = strip_version(accession)
        if cleaned != accession:
            sequence = fetch_uniprot_sequence(session, cleaned)
            if sequence is not None:
                return sequence, "uniprot_fallback"
    return None, None


def load_sequence_cache(path: Path) -> Dict[str, Dict[str, Optional[str]]]:
    if not path.exists():
        return {}
    return json.loads(path.read_text())


def save_sequence_cache(cache: Dict[str, Dict[str, Optional[str]]], path: Path) -> None:
    path.write_text(json.dumps(cache, indent=2, sort_keys=True))


def sanitize_epitope(epitope: str) -> str:
    return AA_ONLY_RE.sub("", str(epitope).upper())


def locate_all_occurrences(sequence: str, peptide: str) -> List[int]:
    if not peptide:
        return []
    starts = []
    start = 0
    while True:
        idx = sequence.find(peptide, start)
        if idx == -1:
            break
        starts.append(idx + 1)
        start = idx + 1
    return starts


def compute_binary_metrics(tp: np.ndarray, fp: np.ndarray, positives: int) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    precision = np.divide(tp, tp + fp, out=np.zeros_like(tp, dtype=float), where=(tp + fp) > 0)
    recall = np.divide(tp, positives, out=np.zeros_like(tp, dtype=float), where=positives > 0)
    f1 = np.divide(
        2 * precision * recall,
        precision + recall,
        out=np.zeros_like(precision, dtype=float),
        where=(precision + recall) > 0,
    )
    return precision, recall, f1


In [19]:
df = pd.read_csv(DATA_CSV)
if "Unnamed: 0" in df.columns:
    df = df.drop(columns=["Unnamed: 0"])

schema_map = {
    "Epitope - Object Type": "Epitope Type",
    "Epitope - Name": "epitope",
    "Epitope - Starting Position": "start",
    "Epitope - Ending Position": "end",
    "Epitope - Molecule Parent": "protein",
    "Epitope - Source Molecule IRI": "uniprot",
    "Epitope - Species": "species",
}
df = df.rename(columns={k: v for k, v in schema_map.items() if k in df.columns})


required_columns = ["Epitope Type", "epitope", "start", "end", "protein", "uniprot", "species"]
missing_columns = [col for col in required_columns if col not in df.columns]
if missing_columns:
    raise ValueError(f"Missing expected columns: {missing_columns}")

def parse_source_accession(value: object) -> object:
    if pd.isna(value):
        return np.nan
    value = str(value).strip().rstrip("/")
    if not value:
        return np.nan
    if value.startswith("http://") or value.startswith("https://"):
        value = value.split("/")[-1]
    return value

df = df.copy()
df["source_accession"] = df["uniprot"].map(parse_source_accession)
df["normalized_accession"] = df["source_accession"].map(strip_version)
df["epitope_clean"] = df["epitope"].map(sanitize_epitope)
df["start"] = pd.to_numeric(df["start"], errors="coerce").astype("Int64")
df["end"] = pd.to_numeric(df["end"], errors="coerce").astype("Int64")
df["accession_source_guess"] = df["source_accession"].map(classify_accession)

display(df.head())
print(f"Rows: {len(df):,}")
print(f"Unique accessions: {df['source_accession'].nunique(dropna=True):,}")
print(df['accession_source_guess'].value_counts(dropna=False))


,Epitope ID - IEDB IRI,Epitope Type,epitope,start,end,Epitope - Source Molecule,uniprot,protein,species,source_accession,normalized_accession,epitope_clean,accession_source_guess
0,http://www.iedb.org/epitope/49,Linear peptide,AAALPGKCGV,66,75,pru p 1,http://www.ncbi.nlm.nih.gov/protein/CAB96876.2,Pru p 3,Prunus persica,CAB96876.2,CAB96876,AAALPGKCGV,ncbi
1,http://www.iedb.org/epitope/100,Linear peptide,AAEAACFK,13,20,Parvalbumin beta,http://www.ncbi.nlm.nih.gov/protein/P02622.1,Parvalbumin beta 2,Gadus morhua,P02622.1,P02622,AAEAACFK,uniprot
2,http://www.iedb.org/epitope/106,Linear peptide,AAEGATPEAKYD,122,133,Pollen allergen Lol p VA,https://www.uniprot.org/uniprot/Q9XF24.1,Lol p 5,Lolium perenne,Q9XF24.1,Q9XF24,AAEGATPEAKYD,uniprot
3,http://www.iedb.org/epitope/195,Linear peptide,AAHASARQQWELQGD,16,30,conglutin-7 precursor [Arachis hypogaea],http://www.ncbi.nlm.nih.gov/protein/NP_0013923...,Ara h 2,Arachis hypogaea,NP_001392340.1,NP_001392340,AAHASARQQWELQGD,ncbi
4,http://www.iedb.org/epitope/303,Linear peptide,AALTKAITAMTQA,251,263,Pollen allergen Lol p VA,https://www.uniprot.org/uniprot/Q9XF24.1,Lol p 5,Lolium perenne,Q9XF24.1,Q9XF24,AALTKAITAMTQA,uniprot


Rows: 5,252
Unique accessions: 478
accession_source_guess
uniprot    2672
ncbi       2580
Name: count, dtype: int64


In [14]:
len(df['protein'].unique())

283

In [4]:
sequence_cache = load_sequence_cache(SEQUENCE_CACHE_PATH)
session = build_session()

unique_accessions = sorted(df["source_accession"].dropna().astype(str).str.strip().unique())
to_fetch = [acc for acc in unique_accessions if acc not in sequence_cache]

for accession in tqdm(to_fetch, desc="Fetching protein sequences"):
    try:
        sequence, source = fetch_sequence(session, accession)
        sequence_cache[accession] = {
            "sequence": sequence,
            "fetch_source": source,
            "normalized_accession": strip_version(accession),
        }
    except Exception as exc:
        sequence_cache[accession] = {
            "sequence": None,
            "fetch_source": None,
            "normalized_accession": strip_version(accession),
            "error": str(exc),
        }
    if len(sequence_cache) % 100 == 0:
        save_sequence_cache(sequence_cache, SEQUENCE_CACHE_PATH)
        time.sleep(0.1)

save_sequence_cache(sequence_cache, SEQUENCE_CACHE_PATH)

seq_df = pd.DataFrame.from_dict(sequence_cache, orient="index").reset_index().rename(columns={"index": "source_accession"})
seq_df["sequence_length"] = seq_df["sequence"].map(lambda x: len(x) if isinstance(x, str) else np.nan)
display(seq_df.head())
print(seq_df["fetch_source"].value_counts(dropna=False))
print(f"Fetched sequences: {seq_df['sequence'].notna().sum():,} / {len(seq_df):,}")


Fetching protein sequences: 0it [00:00, ?it/s]

,source_accession,fetch_source,normalized_accession,sequence,error,sequence_length
0,0401173A,ncbi,0401173A,GHRPLDKKREEAPSLRPAPPPISGGGYRARPAKAAATQKKVERKAP...,NaN,447.0
1,0501217A,ncbi,0501217A,EAQITGRPEWIWLALGTALMGLGTLYFLVKGMGVSDPDAKKFYAIT...,NaN,247.0
2,0704313A,ncbi,0704313A,MNPNQKIITIGSICLVVGLISLILQIGNIISIWISHSIQTGSQNHT...,NaN,454.0
3,0705172A,ncbi,0705172A,GSIGAASMEFCFDVFKELKVHHANENIFYCPIAIMSALAMVYLGAK...,NaN,385.0
4,0804188A,ncbi,0804188A,ADNRDPASDQMKHWKEQRAAQKPDVLTTGGGNPVGDKLNSLTVGPR...,NaN,506.0


fetch_source
ncbi             14891
uniprot           2850
None               389
ncbi_fallback       36
Name: count, dtype: int64
Fetched sequences: 17,777 / 18,166


In [5]:
seq_df.to_csv("curated_b_cell_epitope_allergic_only_all_host_with_sequence.csv")


In [20]:
seq_df = pd.read_csv('curated_b_cell_epitope_allergic_only_all_host_with_sequence.csv')

In [24]:
seq_df.shape

(18166, 7)

In [22]:
df_valid = df.merge(seq_df[["source_accession", "sequence", "fetch_source", "sequence_length"]], on="source_accession", how="left")
df_valid.to_csv(MERGED_EPITOPE_SEQUENCE_PATH, index=False)
print(f"Merged epitope + sequence table saved to: {MERGED_EPITOPE_SEQUENCE_PATH}")

provided_start = df_valid["start"].astype("Int64")
provided_end = df_valid["end"].astype("Int64")
has_coords = provided_start.notna() & provided_end.notna()
has_sequence = df_valid["sequence"].notna()

def extract_provided_substring(row: pd.Series) -> Optional[str]:
    sequence = row["sequence"]
    start = row["start"]
    end = row["end"]
    if not isinstance(sequence, str) or pd.isna(start) or pd.isna(end):
        return None
    start = int(start)
    end = int(end)
    if start < 1 or end < start or end > len(sequence):
        return None
    return sequence[start - 1:end]


df_valid["provided_substring"] = df_valid.apply(extract_provided_substring, axis=1)
df_valid["provided_match"] = df_valid["provided_substring"].fillna("") == df_valid["epitope_clean"]
df_valid["occurrence_starts"] = [locate_all_occurrences(seq, pep) if isinstance(seq, str) else [] for seq, pep in zip(df_valid["sequence"], df_valid["epitope_clean"])]
df_valid["n_occurrences"] = df_valid["occurrence_starts"].map(len)

def assign_validation_status(row: pd.Series) -> str:
    if not isinstance(row["sequence"], str):
        return "missing_full_sequence"
    if not row["epitope_clean"]:
        return "empty_epitope"
    if pd.isna(row["start"]) or pd.isna(row["end"]):
        if row["n_occurrences"] == 1:
            return "rescued_unique_search"
        if row["n_occurrences"] > 1:
            return "missing_coordinates_multiple_matches"
        return "no_match_in_full_sequence"
    start = int(row["start"])
    end = int(row["end"])
    if start < 1 or end < start or end > len(row["sequence"]):
        if RESCUE_UNIQUE_MISMATCHES and row["n_occurrences"] == 1:
            return "rescued_unique_search"
        return "out_of_bounds_coordinates"
    if row["provided_match"]:
        return "exact_coordinate_match"
    if row["n_occurrences"] == 0:
        return "no_match_in_full_sequence"
    if RESCUE_UNIQUE_MISMATCHES and row["n_occurrences"] == 1:
        return "rescued_unique_search"
    return "coordinate_mismatch_multiple_matches"


df_valid["validation_status"] = df_valid.apply(assign_validation_status, axis=1)

def resolved_interval(row: pd.Series) -> Tuple[Optional[int], Optional[int]]:
    status = row["validation_status"]
    peptide = row["epitope_clean"]
    if status == "exact_coordinate_match":
        return int(row["start"]), int(row["end"])
    if status == "rescued_unique_search":
        start = row["occurrence_starts"][0]
        end = start + len(peptide) - 1
        return start, end
    return None, None


resolved = df_valid.apply(resolved_interval, axis=1)
df_valid["resolved_start"] = [item[0] for item in resolved]
df_valid["resolved_end"] = [item[1] for item in resolved]
df_valid["is_usable_row"] = df_valid["validation_status"].isin(["exact_coordinate_match", "rescued_unique_search"])
df_valid.to_csv(VALIDATION_PATH, index=False)

print(df_valid["validation_status"].value_counts(dropna=False).to_string())
display(df_valid.loc[~df_valid["is_usable_row"], ["source_accession", "epitope", "start", "end", "validation_status", "n_occurrences"]].head(20))
print(f"Saved validation table to: {VALIDATION_PATH}")


Merged epitope + sequence table saved to: /Users/anxiongsong/XAllergen/analysis_outputs/curated_b_cell_epitope_table_allergic_only_all_host_with_sequence.csv
validation_status
exact_coordinate_match                  5187
no_match_in_full_sequence                 44
rescued_unique_search                     16
missing_full_sequence                      4
coordinate_mismatch_multiple_matches       1


,source_accession,epitope,start,end,validation_status,n_occurrences
269,CAQ68249.1,GCHGSEPCIIHRGK + ACET(G1),20,33,no_match_in_full_sequence,0
336,CAQ68249.1,HEIKKVLVPGCHGS + ACET(H1),11,24,no_match_in_full_sequence,0
348,CAQ68249.1,HYMKCPLVKGQQYD + ACET(H1),74,87,no_match_in_full_sequence,0
425,Q941R0,KIQRDEDSYE,52,61,no_match_in_full_sequence,0
702,Q941R0,QRCDLDVESGG,146,156,no_match_in_full_sequence,0
817,Q941R0,SQDPYSPSPY,68,77,no_match_in_full_sequence,0
934,CAQ68249.1,VPGIDPNACHYMKC + ACET(V1),65,78,no_match_in_full_sequence,0
976,Q941R0,YERDPYSPSQ,60,69,no_match_in_full_sequence,0
1212,P00711.2,"ALCSEKLDQWLCEKL + MCM(C3, C12)",128,142,no_match_in_full_sequence,0
1223,P00711.2,GYGGVSLPEWVCTTFHTSGYDTQAIVQNNDSTEYGLFQINNK + M...,36,77,no_match_in_full_sequence,0


Saved validation table to: /Users/anxiongsong/XAllergen/analysis_outputs/b_cell_allergic_epitope_sequence_validation.csv


In [23]:
df_valid.shape

(5252, 24)

In [7]:
usable_rows = df_valid.loc[df_valid["is_usable_row"]].copy()

protein_to_sequence: Dict[str, str] = {}
protein_to_labels: Dict[str, np.ndarray] = {}
protein_to_intervals: Dict[str, List[Tuple[int, int]]] = defaultdict(list)
protein_rows = []

for accession, group in usable_rows.groupby("source_accession", sort=True):
    sequence = group["sequence"].iloc[0]
    if not isinstance(sequence, str):
        continue
    if len(sequence) > MAX_SEQUENCE_LENGTH:
        protein_rows.append({
            "source_accession": accession,
            "sequence_length": len(sequence),
            "usable": False,
            "skip_reason": f"sequence_length_gt_{MAX_SEQUENCE_LENGTH}",
            "n_epitope_rows": len(group),
            "n_epitope_intervals": np.nan,
            "n_epitope_residues": np.nan,
        })
        continue

    labels = np.zeros(len(sequence), dtype=np.uint8)
    intervals = []
    for row in group.itertuples(index=False):
        start = int(row.resolved_start)
        end = int(row.resolved_end)
        labels[start - 1:end] = 1
        intervals.append((start, end))

    protein_to_sequence[accession] = sequence
    protein_to_labels[accession] = labels
    protein_to_intervals[accession] = intervals
    protein_rows.append({
        "source_accession": accession,
        "sequence_length": len(sequence),
        "usable": True,
        "skip_reason": None,
        "n_epitope_rows": len(group),
        "n_epitope_intervals": len(intervals),
        "n_epitope_residues": int(labels.sum()),
    })

protein_summary = pd.DataFrame(protein_rows).sort_values(["usable", "n_epitope_rows", "sequence_length"], ascending=[False, False, False])
protein_summary.to_csv(PROTEIN_SUMMARY_PATH, index=False)
display(protein_summary.head())
print(f"Proteins retained for ESM+SAE analysis: {protein_summary['usable'].sum():,}")
print(f"Protein summary saved to: {PROTEIN_SUMMARY_PATH}")


,source_accession,sequence_length,usable,skip_reason,n_epitope_rows,n_epitope_intervals,n_epitope_residues
290,P01005.1,210,True,None,247,247.0,186.0
141,ABW98936.1,214,True,None,183,183.0,199.0
359,P43238.1,626,True,None,175,175.0,589.0
306,P02754.3,178,True,None,144,144.0,162.0
62,AAA48998.1,386,True,None,133,133.0,386.0


Proteins retained for ESM+SAE analysis: 462
Protein summary saved to: /Users/anxiongsong/XAllergen/analysis_outputs/b_cell_allergic_epitope_protein_summary_layer_4.csv


In [8]:
embedder = ESM(model_name=MODEL_NAME, device=device)
sae = load_sae_from_hf(plm_model="esm2-8m", plm_layer=SAE_LAYER)
sae = sae.to(device)
sae.eval()

dict_size = sae.dict_size
threshold_to_index = {thr: idx for idx, thr in enumerate(FEATURE_THRESHOLDS)}
n_thresholds = len(FEATURE_THRESHOLDS)

tp = np.zeros((n_thresholds, dict_size), dtype=np.int64)
fp = np.zeros((n_thresholds, dict_size), dtype=np.int64)
feature_on_total = np.zeros((n_thresholds, dict_size), dtype=np.int64)
segment_hits = np.zeros((n_thresholds, dict_size), dtype=np.int64)
positive_residue_total = 0
positive_segment_total = 0

protein_ids = [pid for pid, row in protein_summary.set_index("source_accession").iterrows() if bool(row["usable"])]

for batch_start in tqdm(range(0, len(protein_ids), BATCH_SIZE), desc="Running ESM2 + SAE"):
    batch_ids = protein_ids[batch_start:batch_start + BATCH_SIZE]
    batch_sequences = [protein_to_sequence[pid] for pid in batch_ids]
    embed_result = embedder.extract_embeddings_with_boundaries(batch_sequences, layer=SAE_LAYER, batch_size=BATCH_SIZE)
    batch_embeddings = embed_result["embeddings"]
    boundaries = embed_result["boundaries"]
    batch_embeddings = batch_embeddings.to(device)
    batch_features = sae.encode(batch_embeddings).detach().cpu()

    for pid, (start_idx, end_idx) in zip(batch_ids, boundaries):
        features = batch_features[start_idx:end_idx]
        labels = torch.from_numpy(protein_to_labels[pid].astype(np.bool_))
        non_epitope_mask = ~labels
        intervals = protein_to_intervals[pid]

        positive_residue_total += int(labels.sum().item())
        positive_segment_total += len(intervals)

        for thr_idx, threshold in enumerate(FEATURE_THRESHOLDS):
            feature_on = features > threshold
            feature_on_total[thr_idx] += feature_on.sum(dim=0).numpy().astype(np.int64)
            tp[thr_idx] += feature_on[labels].sum(dim=0).numpy().astype(np.int64)
            fp[thr_idx] += feature_on[non_epitope_mask].sum(dim=0).numpy().astype(np.int64)

            if intervals:
                interval_hits = torch.zeros(feature_on.shape[1], dtype=torch.bool)
                for start_pos, end_pos in intervals:
                    interval_hits |= feature_on[start_pos - 1:end_pos].any(dim=0)
                segment_hits[thr_idx] += interval_hits.numpy().astype(np.int64)

print(f"Total positive residues: {positive_residue_total:,}")
print(f"Total positive intervals: {positive_segment_total:,}")
print(f"Number of SAE features: {dict_size:,}")


Loading configs from /Users/anxiongsong/.cache/huggingface/hub/models--Elana--InterPLM-esm2-8m/snapshots/81d2429cd9dae7175f1dcd8b4c649a20cdc06c8c/layer_4/config.yaml
Loaded data type: <class 'interplm.train.configs.TrainingRunConfig'>
Data keys: Not a dict


Running ESM2 + SAE:   0%|          | 0/58 [00:00<?, ?it/s]

Total positive residues: 34,692
Total positive intervals: 5,160
Number of SAE features: 10,240


In [9]:
records = []
for thr_idx, threshold in enumerate(FEATURE_THRESHOLDS):
    precision, recall, f1 = compute_binary_metrics(tp[thr_idx], fp[thr_idx], positive_residue_total)
    segment_recall = np.divide(
        segment_hits[thr_idx],
        positive_segment_total,
        out=np.zeros_like(segment_hits[thr_idx], dtype=float),
        where=positive_segment_total > 0,
    )
    for feature_idx in range(dict_size):
        records.append({
            "layer": SAE_LAYER,
            "threshold": threshold,
            "feature": feature_idx,
            "feature_on_residues": int(feature_on_total[thr_idx, feature_idx]),
            "tp": int(tp[thr_idx, feature_idx]),
            "fp": int(fp[thr_idx, feature_idx]),
            "precision": float(precision[feature_idx]),
            "recall": float(recall[feature_idx]),
            "f1": float(f1[feature_idx]),
            "segment_hit_count": int(segment_hits[thr_idx, feature_idx]),
            "segment_recall": float(segment_recall[feature_idx]),
        })

metrics_df = pd.DataFrame.from_records(records)
metrics_df = metrics_df.sort_values(["f1", "segment_recall", "precision"], ascending=[False, False, False])
metrics_df.to_csv(METRICS_PATH, index=False)

threshold_summary = (
    metrics_df.groupby("threshold", as_index=False)
    .agg(
        mean_precision=("precision", "mean"),
        mean_recall=("recall", "mean"),
        mean_f1=("f1", "mean"),
        max_f1=("f1", "max"),
        mean_segment_recall=("segment_recall", "mean"),
        max_segment_recall=("segment_recall", "max"),
        n_features_with_any_tp=("tp", lambda s: int((s > 0).sum())),
    )
    .sort_values(["mean_f1", "mean_segment_recall", "max_f1"], ascending=[False, False, False])
)
threshold_summary.to_csv(THRESHOLD_SUMMARY_PATH, index=False)

best_threshold = float(threshold_summary.iloc[0]["threshold"])
print(f"Thresholds tested: {FEATURE_THRESHOLDS}")
print(f"Threshold selected for summary tables: {best_threshold}")
print(f"Saved feature metrics to: {METRICS_PATH}")
print(f"Saved threshold summary to: {THRESHOLD_SUMMARY_PATH}")
display(threshold_summary)


Thresholds tested: [0.15, 0.3, 0.5, 0.6, 0.8]
Threshold selected for summary tables: 0.15
Saved feature metrics to: /Users/anxiongsong/XAllergen/analysis_outputs/b_cell_allergic_epitope_feature_metrics_layer_4.csv
Saved threshold summary to: /Users/anxiongsong/XAllergen/analysis_outputs/b_cell_allergic_epitope_threshold_summary_layer_4.csv


,threshold,mean_precision,mean_recall,mean_f1,max_f1,mean_segment_recall,max_segment_recall,n_features_with_any_tp
0,0.15,0.051893,0.002182,0.003568,0.223720,0.002261,0.081589,1699
1,0.30,0.039285,0.000708,0.001261,0.136645,0.001142,0.074806,1250
2,0.50,0.027187,0.000245,0.000435,0.131461,0.000435,0.071318,837
3,0.60,0.023331,0.000134,0.000244,0.101748,0.000265,0.067442,641
4,0.80,0.012967,0.000006,0.000012,0.007388,0.000026,0.016473,253


In [10]:
best_threshold_df = metrics_df.loc[metrics_df["threshold"] == best_threshold].copy()
display(best_threshold_df.head(25))

print("Top features at the selected threshold")
display(
    best_threshold_df.loc[:, ["feature", "precision", "recall", "f1", "segment_recall", "feature_on_residues", "tp", "fp"]]
    .head(25)
)


,layer,threshold,feature,feature_on_residues,tp,fp,precision,recall,f1,segment_hit_count,segment_recall
4375,4,0.15,4375,13386,5378,8008,0.401763,0.155021,0.223720,345,0.066860
5025,4,0.15,5025,20308,5818,14490,0.286488,0.167704,0.211564,406,0.078682
8432,4,0.15,8432,14348,5008,9340,0.349038,0.144356,0.204241,371,0.071899
7846,4,0.15,7846,16542,5000,11542,0.302261,0.144125,0.195183,373,0.072287
2736,4,0.15,2736,15929,4647,11282,0.291732,0.133950,0.183600,407,0.078876
5838,4,0.15,5838,16171,4612,11559,0.285202,0.132941,0.181350,408,0.079070
1119,4,0.15,1119,15321,4380,10941,0.285882,0.126254,0.175154,377,0.073062
2163,4,0.15,2163,16523,4395,12128,0.265993,0.126686,0.171629,410,0.079457
7865,4,0.15,7865,14626,4168,10458,0.284972,0.120143,0.169026,373,0.072287
6162,4,0.15,6162,16699,4343,12356,0.260075,0.125187,0.169018,416,0.080620


Top features at the selected threshold


,feature,precision,recall,f1,segment_recall,feature_on_residues,tp,fp
4375,4375,0.401763,0.155021,0.223720,0.066860,13386,5378,8008
5025,5025,0.286488,0.167704,0.211564,0.078682,20308,5818,14490
8432,8432,0.349038,0.144356,0.204241,0.071899,14348,5008,9340
7846,7846,0.302261,0.144125,0.195183,0.072287,16542,5000,11542
2736,2736,0.291732,0.133950,0.183600,0.078876,15929,4647,11282
5838,5838,0.285202,0.132941,0.181350,0.079070,16171,4612,11559
1119,1119,0.285882,0.126254,0.175154,0.073062,15321,4380,10941
2163,2163,0.265993,0.126686,0.171629,0.079457,16523,4395,12128
7865,7865,0.284972,0.120143,0.169026,0.072287,14626,4168,10458
6162,6162,0.260075,0.125187,0.169018,0.080620,16699,4343,12356


## Outputs

The notebook writes these files:

- `analysis_outputs/b_cell_epitope_sequence_validation.csv`: row-level validation and rescue status.
- `analysis_outputs/b_cell_epitope_protein_summary_layer_4.csv`: per-protein inclusion and label coverage summary.
- `analysis_outputs/b_cell_epitope_feature_metrics_layer_4.csv`: per-feature precision, recall, F1, and segment recall for every tested threshold.
- `analysis_outputs/b_cell_epitope_threshold_summary_layer_4.csv`: threshold-level summary. The notebook prints the threshold selected for summary tables.

If you want to sweep other InterPLM layers, change `SAE_LAYER` to one of the pretrained ESM2-8M layers: `1, 2, 3, 4, 5, 6`.